In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Creating Agents with the OpenAI Agents SDK
---

# Using a Local LLM via LiteLLM and Ollama
* Agents SDK programming model while changing the model provider
* Local models are useful for 
    * Privacy-sensitive drafts 
    * Offline experimentation
    * Cost saving
* Local model quality, speed and context limits vary significantly by model and hardware
    * Testing on my MacBook Pro M2 MAX with 96GB of RAM was SLOW
    * Tool use was flaky
    * Not as good at following instructions as OpenAI's models, even when using open-source "reasoning" models
    * For production workflows, test the exact model you plan to use with realistic inputs
* **LiteLLM** lets the Agents SDK call many model providers through one interface
* **`LitellmModel`** is a model adapter you can pass to `Agent(model=...)`
* **Ollama** runs open-weights models locally
* This example sends a local transcript file to a local model and asks for a concise summary
* **`set_tracing_disabled(True)`** avoids OpenAI tracing when running through a non-OpenAI provider

---

## Setup — This Notebook Uses Its Own Environment
* LiteLLM releases require `openai` 2.x; the Agents SDK 0.22 used by the rest of this course requires `openai` 3.x
    * They can't share one environment, so this demo has a separate one: **`deitel-openai-litellm`**
* Create it once from the course folder:
    * **Anaconda/macOS:** `bash setup/setup_litellm_mac.sh`
    * **Anaconda/Windows:** `setup\setup_litellm_windows.bat`
    * **pip/macOS or Linux:** `bash setup/setup_litellm_pip_mac.sh`
    * **pip/Windows:** `setup\setup_litellm_pip_windows.bat`
* In JupyterLab, choose the **Python (deitel-openai-litellm)** kernel for this notebook only (Kernel > Change Kernel...)
* Install Ollama from https://ollama.com/download
* Pull a model you want to use:
    >```bash
    >ollama pull deepseek-r1:14b
    >```
* Run
    >```bash
    >ollama serve
    >```

---


## Imports and Constants

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel

set_tracing_disabled(disabled=True) # tracing works only with OpenAI hosted models

MODEL = 'ollama/deepseek-r1:14b' # model to run locally
APIKEY = 'ollama'  # Ollama ignores the key; LiteLLM requires a non-empty value.
API_BASE = 'http://localhost:11434'

---
## Load the Transcript

In [ ]:
TRANSCRIPT_PATH = Path('resources') / 'transcript.txt'
transcript = TRANSCRIPT_PATH.read_text(encoding='utf-8')

print(f'Loaded {TRANSCRIPT_PATH} ({len(transcript):,} characters)')

---
## Agent

* Same `Agent` and `Runner.run(...)` pattern as the OpenAI examples
* Model-specific piece is `LitellmModel(...)`


In [ ]:
summarizer_agent = Agent(
    name='Local Transcript Summarizer',
    model=LitellmModel(
        model=MODEL,
        api_key=APIKEY,
        base_url=API_BASE
    ),
    instructions="""
        You summarize technical presentation transcripts. Given a 
        transcript, create a summary abstract paragraph and key-points
        bullet list. Use straightforward sentences. Spell technical 
        features correctly. Avoid abbreviations. Do not refer to the 
        speaker. Do not invent details that are not present in the 
        transcript. If the transcript is unclear, say so. 
        
        DO NOT include original transcript in your response. 
    """
)

---
## Run the Summary Agent
* The transcript is passed as ordinary input text

In [ ]:
prompt = f"""
Summarize the following transcript.

Transcript:
{transcript}
"""

result = await Runner.run(summarizer_agent, prompt)
display(Markdown(result.final_output))

---
## References

* [Agents SDK LiteLLM models](https://openai.github.io/openai-agents-python/models/litellm/)
* [LiteLLM Ollama provider](https://docs.litellm.ai/docs/providers/ollama)
* [Ollama](https://ollama.com/)

---


© 2026 by Deitel & Associates, Inc. All Rights Reserved.